# Drought in Control — Ekstraksi 45 Tahun Hujan CHIRPS v2.0 (1981–2025) untuk 3D Pin-Field Relief

Notebook ini menghasilkan berkas biner **`PIN1` (`.pinbin`)** berisi **SPI-12 tahunan asli dari satelit `UCSB-CHG/CHIRPS/DAILY` (1981–2025)** di atas grid daratan Indonesia (`0.13°` / `0.045°` / `0.025°`), siap langsung diimpor ke **Studio 3D Relief (`agri.transbulent.id/kekeringan/relief/`)**.

### Pengaman Biaya & Kuota Colab
- Watchdog latar belakang + `finally:` otomatis memanggil `google.colab.runtime.unassign()` bila idle >= **180 detik** atau tepat setelah `.pinbin` selesai diunduh.

In [ ]:
#@title 1. Konfigurasi Wilayah & Watchdog Auto-Shutdown 3 Menit
import threading, time

REGION = "jawa"  #@param ["indonesia", "jawa", "jateng", "sumatera", "ntt_bali", "kalimantan", "sulawesi"]
START_YEAR = 1981  #@param {type:"integer"}
END_YEAR = 2025  #@param {type:"integer"}
GEE_PROJECT_ID = ""  #@param {type:"string"}
AUTO_SHUTDOWN_AFTER_EXPORT = True  #@param {type:"boolean"}
IDLE_LIMIT_SEC = 180  #@param {type:"integer"}

_LAST_ACTIVITY = [time.time()]
_STOP_WATCHDOG = [False]

def touch_activity():
    _LAST_ACTIVITY[0] = time.time()

def shutdown_colab_runtime(reason="Selesai / idle"):
    print(f"[AUTO-SHUTDOWN] Mematikan runtime Colab ({reason})...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print(f"[INFO] Tidak di Colab runtime atau sudah mati: {exc}")

def _watchdog_loop():
    while not _STOP_WATCHDOG[0]:
        time.sleep(10)
        if time.time() - _LAST_ACTIVITY[0] >= IDLE_LIMIT_SEC:
            shutdown_colab_runtime(f"idle >= {IDLE_LIMIT_SEC}s")
            break

threading.Thread(target=_watchdog_loop, daemon=True).start()
print(f"Preset: {REGION} · Rentang: {START_YEAR}-{END_YEAR} · Auto-off={AUTO_SHUTDOWN_AFTER_EXPORT} ({IDLE_LIMIT_SEC}s)")

In [ ]:
#@title 2. Ekstraksi CHIRPS v2.0 45 Tahun dari Google Earth Engine & Ekspor .pinbin
import json, math, struct, urllib.request

GRID_URL = f"https://agri.transbulent.id/kekeringan/relief-data/grid-{REGION}.json"
OUT_PINBIN = f"{REGION}-chirps-{START_YEAR}-{END_YEAR}.pinbin"

def quantize_i8(val, smin=-3.0, smax=3.0):
    if val is None or math.isnan(val) or math.isinf(val):
        return -128
    t = max(0.0, min(1.0, (val - smin) / (smax - smin)))
    return max(-127, min(127, int(round(t * 254.0 - 127.0))))

try:
    touch_activity()
    with urllib.request.urlopen(GRID_URL, timeout=20) as r:
        grid = json.loads(r.read().decode("utf-8"))
    lons, lats = grid["lons"], grid["lats"]
    step_deg = float(grid["stepDeg"])
    scale_m = max(5000, int(round(step_deg * 111320)))
    print(f"Grid dimuat: {len(lons)} pin darat ({grid['spacingKm']} km)")

    import ee
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID.strip()) if GEE_PROJECT_ID.strip() else ee.Initialize()
    touch_activity()

    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    years = list(range(int(START_YEAR), int(END_YEAR) + 1))
    annual_imgs = [
        chirps.filterDate(f"{y}-01-01", f"{y + 1}-01-01").select("precipitation").sum().rename(f"y{y}")
        for y in years
    ]
    log_stack = ee.Image.cat(annual_imgs).add(1.0).log()
    spi_stack = (
        log_stack.subtract(log_stack.reduce(ee.Reducer.mean()))
        .divide(log_stack.reduce(ee.Reducer.stdDev()).max(0.05))
        .clamp(-3.0, 3.0)
    )

    frames = [[None] * len(lons) for _ in years]
    chunk = 1500
    for start in range(0, len(lons), chunk):
        touch_activity()
        end = min(len(lons), start + chunk)
        fc = ee.FeatureCollection([
            ee.Feature(ee.Geometry.Point([lons[i], lats[i]]), {"pid": i}) for i in range(start, end)
        ])
        sampled = spi_stack.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=scale_m, tileScale=4).getInfo()
        for feat in sampled.get("features", []):
            props = feat.get("properties", {})
            pid = int(props["pid"])
            for yi, y in enumerate(years):
                val = props.get(f"y{y}")
                frames[yi][pid] = float(val) if val is not None else None
        print(f"Sampling {end}/{len(lons)} pin selesai...")

    meta = {
        "region": grid["id"],
        "label": grid["label"],
        "metric": "spi3",
        "cadence": "annual_spi12",
        "source": f"UCSB-CHG/CHIRPS/DAILY ({START_YEAR}-{END_YEAR})",
        "is_simulated": False,
        "spacing_km": grid["spacingKm"],
        "step_deg": step_deg,
        "bbox": grid["bbox"],
        "pin_count": len(lons),
        "time_count": len(years),
        "time_labels": [str(y) for y in years],
        "scale_min": -3.0,
        "scale_max": 3.0,
        "anchors": grid.get("anchors", []),
    }
    meta_bytes = json.dumps(meta, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    buf = bytearray(b"PIN1" + struct.pack("<I", len(meta_bytes)) + meta_bytes)
    buf.extend(struct.pack(f"<{len(lons)}f", *lons))
    buf.extend(struct.pack(f"<{len(lats)}f", *lats))
    for fr in frames:
        buf.extend(struct.pack(f"<{len(lons)}b", *[quantize_i8(v) for v in fr]))

    with open(OUT_PINBIN, "wb") as f:
        f.write(buf)
    print(f"[SELESAI] {OUT_PINBIN} ({len(buf)/1024:.1f} KB). Drop ke panel Studio Ekspor di agri.transbulent.id/kekeringan/relief/!")
    try:
        from google.colab import files
        files.download(OUT_PINBIN)
        time.sleep(10)
    except Exception:
        pass
finally:
    _STOP_WATCHDOG[0] = True
    if AUTO_SHUTDOWN_AFTER_EXPORT:
        shutdown_colab_runtime("ekspor .pinbin selesai")